# Creating fieldmaps
 Tailor this notebook for McNab/Parvizi lab imaging pipeline

In [12]:
import glob
import os
import json
import subprocess
import pandas as pd
from nilearn import image
import nibabel as nib


In [16]:
json_fname = '/Users/heejungj/Library/CloudStorage/Box-Box/MRI in iEEG Patients - Parvizi, McNab, Keller Labs/Images/18119_09032025/006/006_McNab_Parvizi_Brain_25_0_20250903080551_6a.json'
with open(json_fname, 'r') as f:
    json_data = json.load(f)

json_data

{'Modality': 'MR',
 'MagneticFieldStrength': 3,
 'ImagingFrequency': 127.762051,
 'Manufacturer': 'GE',
 'InternalPulseSequenceName': 'EPI',
 'ManufacturersModelName': 'SIGNA Premier',
 'InstitutionName': 'Lucas Center',
 'DeviceSerialNumber': '0000415723SHMR18',
 'StationName': 'No2',
 'BodyPart': 'BRAIN',
 'PatientPosition': 'HFS',
 'ProcedureStepDescription': 'Gradient Echo',
 'SoftwareVersions': '30\\LX\\SIGNA_LX1.MR30.1_R01B_2430.c',
 'MRAcquisitionType': '2D',
 'SeriesDescription': 'epi-RS_run-01',
 'ProtocolName': 'McNab_Parvizi_Brain_25_0',
 'ScanningSequence': 'EP\\RM',
 'SequenceVariant': 'NONE',
 'ScanOptions': 'SAT_GEMS\\EDR_GEMS\\MP_GEMS\\EPI_GEMS\\HYPERBAND_GEMS\\PFF\\FS',
 'PulseSequenceName': 'epigg',
 'ImageType': ['ORIGINAL', 'PRIMARY', 'OTHER', 'MAGNITUDE'],
 'SeriesNumber': 6,
 'AcquisitionTime': '08:21:12.000000',
 'AcquisitionNumber': 1,
 'TriggerDelayTime': 479980,
 'SliceThickness': 2.5,
 'SpacingBetweenSlices': 2.5,
 'SAR': 0.0830134,
 'TablePosition': [0, 0, -

In [ ]:

file_path = r'/Users/heejungj/Library/CloudStorage/Box-Box/MRI in iEEG Patients - Parvizi, McNab, Keller Labs/Images/18119_09032025/nifti/006_McNab_Parvizi_Brain_25_0_20250903080551_6.nii.gz'

img = image.load_img(file_path)
print(img.header)

img = nib.load(file_path)
print(img.header)

<class 'nibabel.nifti1.Nifti1Header'> object, endian='<'
sizeof_hdr      : 348
data_type       : b''
db_name         : b''
extents         : 0
session_error   : 0
regular         : b'r'
dim_info        : 57
dim             : [  4  88  88  50 480   1   1   1]
intent_p1       : 0.0
intent_p2       : 0.0
intent_p3       : 0.0
intent_code     : none
datatype        : int16
bitpix          : 16
slice_start     : 0
pixdim          : [-1.   2.5  2.5  2.5  1.   0.   0.   0. ]
vox_offset      : 0.0
scl_slope       : nan
scl_inter       : nan
slice_end       : 0
slice_code      : unknown
xyzt_units      : 10
cal_max         : 0.0
cal_min         : 0.0
slice_duration  : 0.0
toffset         : 0.0
glmax           : 0
glmin           : 0
descrip         : b'TE=30;Time=82112.000;phase=1;mb=4'
aux_file        : b''
qform_code      : scanner
sform_code      : scanner
quatern_b       : 0.0
quatern_c       : 0.9984834
quatern_d       : 0.055053394
qoffset_x       : 107.089874
qoffset_y       : -98.82157


In [18]:
json_data['TotalReadoutTime']

0.049068

In [38]:
# Helper function to load and parse JSON files
def extract_phase_encoding_data(files, file_type, run_name):
    data = []
    for file_path in sorted(files):
        file_name = os.path.basename(file_path)
        print(file_name)
        try:
            with open(file_path, 'r') as file:
                json_data = json.load(file)
                run_name = file_name.split("_")[2]
                total_readout = json_data['TotalReadoutTime']
                # Optionally, if multiple files are processed, you might want to warn if they differ
                # if total_readout is None:
                #     total_readout = json_data['info']['TotalReadoutTime']
                # elif total_readout != current_total:
                #     print(f"Warning: Different TotalReadoutTime encountered in {file_path}. Using first value.")

                if file_type == "fmap":
                    data.append({
                        "run": run_name,
                        "fmap_PhaseEncodingPolarityGE": json_data['PhaseEncodingPolarityGE'],
                        "fmap_PhaseEncodingDirection": json_data['PhaseEncodingDirection']
                    })
                elif file_type == "func":
                    # task_name = file_name.split("_")[1]
                    data.append({
                        # "task": task_name,
                        "run": run_name,
                        "bold_PhaseEncodingPolarityGE": json_data['PhaseEncodingPolarityGE'],
                        "bold_PhaseEncodingDirection": json_data['PhaseEncodingDirection']
                    })
        except KeyError as e:
            print(f"Missing key {e} in {file_path}. Skipping.")
        except Exception as e:
            print(f"Error processing {file_path}: {e}. Skipping.")
    return data, total_readout


In [33]:
def create_acqparams_per_run_with_opposite(merged_table, output_dir, sub, dim4, DIRECTION_MAPPING):
    for run in merged_table["run"].unique():
        run_data = merged_table[merged_table["run"] == run]

        # Generate main direction
        acqparams = []
        for _, row in run_data.iterrows():
            main_direction = DIRECTION_MAPPING.get(
                row.get("fmap_PhaseEncodingDirection", row.get("bold_PhaseEncodingDirection", "N/A")),
                "0 0 0 0"
            )
            acqparams.append(main_direction)

            # Automatically infer opposite phase encoding
            opposite_direction = " ".join(
                str(-int(value)) if i < 3 else value
                for i, value in enumerate(main_direction.split())
            )

        # Create the repeated rows for main and opposite directions
        repeated_acqparams = ([main_direction] * dim4) + ([opposite_direction] * dim4)

        # Write to the output file
        output_path = os.path.join(output_dir, f"{sub}_{run}_acqparams.txt")
        with open(output_path, "w") as file:
            file.write("\n".join(repeated_acqparams))
        print(f"acqparams.txt created for {run} at: {output_path}")




# baed on encoding direction metadata,
We create a mapping with 
a) 3 columns of phase encoding direction
b) 4th column of total readout time
- total read out time is based on `(number of echoes minus one) * echo spacing`
- In our .json files, the corresponding parameter for 
- number of echo is `AcquisitionMatrixPE`
- echo spacing is `EffectiveEchoSpacing`

- Further information: https://lcni.uoregon.edu/wiki/tags/fmri/#:~:text=The%20total%20readout%20time%20is,applying%20the%20fieldmap%20to%20later.

## single usage case

In [50]:
extract_phase_encoding_data(['/Users/heejungj/Library/CloudStorage/Box-Box/MRI in iEEG Patients - Parvizi, McNab, Keller Labs/Images/18119_09032025/007/007_McNab_Parvizi_Brain_25_0_20250903080551_7.json'], 
                            file_type="fmap",
                            run_name=1)

007_McNab_Parvizi_Brain_25_0_20250903080551_7.json


([{'run': 'Parvizi',
   'fmap_PhaseEncodingPolarityGE': 'Flipped',
   'fmap_PhaseEncodingDirection': 'j'}],
 0.049068)

## NOTE: 
fmap_list should be a list


In [56]:
# Define constants

for sub in ['18119_09032025']: #['sub-001', 'sub-002','sub-003']:
    base_dir = f"/Users/heejungj/Documents/projects_local/iEEGMcNab"
    fmap_path = r'/Users/heejungj/Library/CloudStorage/Box-Box/MRI in iEEG Patients - Parvizi, McNab, Keller Labs/Images/18119_09032025/007/007_McNab_Parvizi_Brain_25_0_20250903080551_7.json' #f"{base_dir}/{sub}/fmap/{sub}_dir-PA_run-*_epi.json"
    func_path = r'/Users/heejungj/Library/CloudStorage/Box-Box/MRI in iEEG Patients - Parvizi, McNab, Keller Labs/Images/18119_09032025/006/006_McNab_Parvizi_Brain_25_0_20250903080551_6a.json' #f"{base_dir}/{sub}/func/{sub}_task-*_run-*_bold.json"
    output_dir = f"{base_dir}/acqparams_per_run/{sub}"

    # Ensure output directory exists
    os.makedirs(output_dir, exist_ok=True)


    # Extract data
    fmap_files = glob.glob(fmap_path)
    func_files = glob.glob(func_path)
    # Remember, this total readout is all from the pepolar sequence, 
    # NOT the functional sequence you will be applying the fieldmap to later
    fmap_data, total_readout = extract_phase_encoding_data(fmap_files, file_type="fmap", run_name=1)
    func_data, NULL = extract_phase_encoding_data(func_files, file_type="func", run_name=1)

    
    DIRECTION_MAPPING = {
        "i": f"1 0 0 {total_readout}",
        "i-": f"-1 0 0 {total_readout}",
        "j": f"0 1 0 {total_readout}",
        "j-": f"0 -1 0 {total_readout}",
        "k": f"0 0 1 {total_readout}",
        "k-": f"0 0 -1 {total_readout}",
    }
    # Convert to DataFrames
    fmap_table = pd.DataFrame(fmap_data)
    func_table = pd.DataFrame(func_data)


    # Merge DataFrames on "Run"
    merged_table = pd.merge(fmap_table, func_table, on="run", how="outer")
    merged_table['sub'] = sub
    # Assume df is your DataFrame
    desired_order = ['sub', 'run', 'bold_PhaseEncodingPolarityGE', 'bold_PhaseEncodingDirection', 'fmap_PhaseEncodingPolarityGE', 'fmap_PhaseEncodingDirection']
    merged_table = merged_table[desired_order]
    os.makedirs('phase_encoding', exist_ok=True)

    merged_table.to_csv(f'./phase_encoding/{sub}_phase_encoding.tsv', sep='\t', header=True, index=False)
    # merged_table
    # Pretty print merged_table
    print(f"\n\n=== Phase Encoding Table for {sub} ===")
    print(merged_table.to_string(index=False, justify='center'))

    # Call the function to generate acqparams
    with open(fmap_files[0], 'r') as file:
        check_dim4 = json.load(file)
    # dim4 = check_dim4['NumberOfTemporalPositions'] #-> 09/06/2025 need to change parameter
    dim4 = int(check_dim4['AcquisitionDuration'] / check_dim4['RepetitionTime'])
    create_acqparams_per_run_with_opposite(merged_table, output_dir, sub, dim4, DIRECTION_MAPPING)


007_McNab_Parvizi_Brain_25_0_20250903080551_7.json
006_McNab_Parvizi_Brain_25_0_20250903080551_6a.json


=== Phase Encoding Table for 18119_09032025 ===
     sub         run   bold_PhaseEncodingPolarityGE bold_PhaseEncodingDirection fmap_PhaseEncodingPolarityGE fmap_PhaseEncodingDirection
18119_09032025 Parvizi          Unflipped                        j-                       Flipped                         j             
acqparams.txt created for Parvizi at: /Users/heejungj/Documents/projects_local/iEEGMcNab/acqparams_per_run/18119_09032025/18119_09032025_Parvizi_acqparams.txt
